# 🏦 Loan Default Prediction Portfolio Project

Welcome to the second portfolio project for **Logistic Regression**. In this notebook, we will predict whether a bank customer will default on their loan.

## 1. Business Problem
A bank wants to minimize its financial risk by identifying customers who are likely to default on their personal loans. A false negative (approving a loan that defaults) costs the bank heavily, while a false positive (denying a good customer) results in lost interest revenue. We need a calibrated probability model to balance these risks.

## 2. Dataset Description
We will generate a synthetic financial dataset.

**Features:**
- `Income`: Annual income of the applicant.
- `CreditScore`: Applicant's credit score (300-850).
- `DebtToIncomeRatio`: Percentage of income going toward debt payments.
- `LoanAmount`: The requested loan amount.
- **Target:** `Default` (1 if default, 0 if fully paid).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

np.random.seed(101)
n_samples = 4000

income = np.random.normal(60000, 20000, n_samples)
credit_score = np.random.normal(650, 80, n_samples)
dti = np.random.uniform(0.1, 0.5, n_samples)
loan_amount = np.random.uniform(5000, 40000, n_samples)

# Logic: Low credit score, high DTI, high loan vs income increases default
z = 10.0 - 0.015 * credit_score + 8.0 * dti + 0.0001 * loan_amount - 0.00005 * income
prob_default = 1 / (1 + np.exp(-z))
default = (np.random.rand(n_samples) < prob_default).astype(int)

df = pd.DataFrame({
    'Income': income,
    'CreditScore': credit_score,
    'DebtToIncomeRatio': dti,
    'LoanAmount': loan_amount,
    'Default': default
})

df.head()

## 3. Exploratory Data Analysis (EDA)
Let's see how Credit Score impacts Default rates.

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x='Default', y='CreditScore', data=df)
plt.title('Credit Score vs Default')

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='DebtToIncomeRatio', hue='Default', element="step")
plt.title('DTI Ratio vs Default')
plt.tight_layout()
plt.show()

## 4. Feature Engineering
We scale the continuous features. We could also engineer a new feature: `Loan_to_Income_Ratio = LoanAmount / Income`.

In [ ]:
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['Income']

X = df.drop('Default', axis=1)
y = df['Default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training & 6. Hyperparameter Tuning
We train Logistic Regression with L2 Penalty.

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {'C': [0.1, 1, 10], 'class_weight': [None, 'balanced']}
model = GridSearchCV(LogisticRegression(max_iter=1000), grid, cv=5, scoring='roc_auc')
model.fit(X_train_scaled, y_train)

print("Best params:", model.best_params_)

## 7. Evaluation & 8. Error Analysis
Since defaults are the minority class, we look closely at recall for class 1.

In [ ]:
best_lr = model.best_estimator_
y_pred = best_lr.predict(X_test_scaled)
y_prob = best_lr.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

## 9. Visualizations & 10. Business Insights
Which factors most strongly predict a default?

In [ ]:
coefs = pd.DataFrame({'Feature': X.columns, 'Importance': best_lr.coef_[0]}).sort_values('Importance')
sns.barplot(x='Importance', y='Feature', data=coefs, orient='h')
plt.title("Drivers of Loan Default")
plt.show()

print("Insights:")
print("1. High DTI (Debt to Income) drastically increases default risk.")
print("2. Higher Credit Scores effectively reduce the risk.")

## 11. Future Improvements
- Use an ensemble model to capture interaction effects between low income and high loan amounts.